# 02 — Mass–Spring–Damper Cube in NVIDIA Warp

## 목표

8개의 mass와 28개의 spring으로 deformable cube를 직접 구현한다.

- 8 point masses
- 12 edge springs
- 12 face diagonals
- 4 body diagonals
- Hooke spring + dashpot damping
- gravity
- penalty ground contact
- 12 edge springs를 actuator로 사용

### 핵심 식

\[
\mathbf{F}_{spring}
=
\left(k(L-L_0)+c(\mathbf{v}_{j}-\mathbf{v}_{i})\cdot\mathbf{n}ight)\mathbf{n}
\]

actuator는 힘을 직접 출력하지 않고 rest length를 바꾼다.

\[
L_0(t)=L_{0,\mathrm{base}}(1+A a_t)
\]

In [ ]:
# Colab 권장: Runtime > Change runtime type > T4 GPU
!nvidia-smi

# Warp 1.17.0의 PyPI Linux wheel은 CUDA 12.9 runtime 기반이라
# Colab의 일반적인 NVIDIA driver에서 호환성이 좋다.
%pip -q install "warp-lang==1.17.0"

import warp as wp
wp.init()
wp.print_diagnostics()

DEVICE = "cuda:0" if wp.is_cuda_available() else "cpu"
print("Selected Warp device:", DEVICE)

## 1. Cube topology

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warp as wp

SIDE = 0.4

def make_cube_topology(side=SIDE, z0=0.45):
    # bit pattern 순서의 8 vertices
    verts = np.array([
        [0,0,0], [1,0,0], [0,1,0], [1,1,0],
        [0,0,1], [1,0,1], [0,1,1], [1,1,1],
    ], dtype=np.float32)

    verts *= side
    verts[:, 0] -= side * 0.5
    verts[:, 1] -= side * 0.5
    verts[:, 2] += z0

    springs = []
    actuator_index = []
    actuator_count = 0

    # 모든 8C2 = 28 pair를 연결:
    # 12 edge + 12 face diagonal + 4 body diagonal
    for i in range(8):
        for j in range(i + 1, 8):
            d = np.linalg.norm(verts[j] - verts[i])
            springs.append((i, j))

            if np.isclose(d, side, atol=1e-5):
                actuator_index.append(actuator_count)
                actuator_count += 1
            else:
                actuator_index.append(-1)

    spring_i = np.array([s[0] for s in springs], dtype=np.int32)
    spring_j = np.array([s[1] for s in springs], dtype=np.int32)
    rest = np.array(
        [np.linalg.norm(verts[j] - verts[i]) for i, j in springs],
        dtype=np.float32
    )

    return verts, spring_i, spring_j, rest, np.array(actuator_index, np.int32)

BASE_X, SPRING_I, SPRING_J, BASE_REST, ACT_IDX = make_cube_topology()

print("particles:", len(BASE_X))
print("springs:", len(SPRING_I))
print("actuated edge springs:", np.sum(ACT_IDX >= 0))

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

def plot_cube_np(x, spring_i=SPRING_I, spring_j=SPRING_J, title="Mass-Spring Cube"):
    fig = plt.figure(figsize=(6, 5))
    ax = fig.add_subplot(111, projection="3d")

    for i, j in zip(spring_i, spring_j):
        p, q = x[i], x[j]
        ax.plot([p[0], q[0]], [p[1], q[1]], [p[2], q[2]], alpha=0.3)

    ax.scatter(x[:,0], x[:,1], x[:,2], s=55)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_zlim(-0.05, 0.9)
    ax.set_title(title)
    plt.show()

plot_cube_np(BASE_X)

## 2. Physics kernels

구조가 작기 때문에 이 notebook에서는 **particle 하나당 thread 하나**를 두고,
각 particle thread가 28개 spring을 훑으며 자신의 force를 계산한다.

이 방식은 atomic operation이 없어 구현이 단순하고,
다음 notebook에서 수천 개 cube로 그대로 확장하기 쉽다.

In [ ]:
@wp.kernel
def compute_force_single(
    x: wp.array(dtype=wp.vec3),
    v: wp.array(dtype=wp.vec3),
    spring_i: wp.array(dtype=wp.int32),
    spring_j: wp.array(dtype=wp.int32),
    rest: wp.array(dtype=float),
    actuator_index: wp.array(dtype=wp.int32),
    action: wp.array(dtype=float),
    force: wp.array(dtype=wp.vec3),
    n_springs: int,
    mass: float,
    k: float,
    c: float,
    act_amp: float,
    ground_k: float,
    ground_c: float,
    friction: float,
):
    p = wp.tid()

    xp = x[p]
    vp = v[p]
    f = wp.vec3(0.0, 0.0, -9.81 * mass)

    # 이 particle과 연결된 모든 spring을 합산한다.
    for s in range(n_springs):
        i = spring_i[s]
        j = spring_j[s]

        if p == i or p == j:
            xi = x[i]
            xj = x[j]
            vi = v[i]
            vj = v[j]

            d = xj - xi
            L = wp.length(d)
            n = d / (L + 1.0e-8)

            L0 = rest[s]
            aidx = actuator_index[s]
            if aidx >= 0:
                L0 = L0 * (1.0 + act_amp * action[aidx])

            rel = wp.dot(vj - vi, n)
            mag = k * (L - L0) + c * rel
            fs = mag * n

            if p == i:
                f = f + fs
            else:
                f = f - fs

    # z=0 penalty ground
    if xp[2] < 0.0:
        f = f + wp.vec3(0.0, 0.0, -ground_k * xp[2])

        if vp[2] < 0.0:
            f = f + wp.vec3(0.0, 0.0, -ground_c * vp[2])

        f = f + wp.vec3(-friction * vp[0], -friction * vp[1], 0.0)

    force[p] = f


@wp.kernel
def integrate_single(
    x: wp.array(dtype=wp.vec3),
    v: wp.array(dtype=wp.vec3),
    force: wp.array(dtype=wp.vec3),
    mass: float,
    dt: float,
):
    p = wp.tid()

    a = force[p] / mass
    v_new = v[p] + a * dt
    x_new = x[p] + v_new * dt

    v[p] = v_new
    x[p] = x_new

## 3. Simulator class

In [ ]:
class WarpCube:
    def __init__(
        self,
        device=DEVICE,
        dt=0.0025,
        substeps=4,
        mass=0.15,
        k=180.0,
        c=2.0,
        act_amp=0.20,
        ground_k=1500.0,
        ground_c=15.0,
        friction=2.0,
    ):
        self.device = device
        self.dt = dt
        self.substeps = substeps
        self.mass = mass
        self.k = k
        self.c = c
        self.act_amp = act_amp
        self.ground_k = ground_k
        self.ground_c = ground_c
        self.friction = friction

        self.n_particles = 8
        self.n_springs = len(SPRING_I)
        self.n_act = int(np.sum(ACT_IDX >= 0))

        self.spring_i = wp.array(SPRING_I, dtype=wp.int32, device=device)
        self.spring_j = wp.array(SPRING_J, dtype=wp.int32, device=device)
        self.rest = wp.array(BASE_REST, dtype=float, device=device)
        self.actuator_index = wp.array(ACT_IDX, dtype=wp.int32, device=device)

        self.x = wp.array(BASE_X, dtype=wp.vec3, device=device)
        self.v = wp.zeros(8, dtype=wp.vec3, device=device)
        self.force = wp.zeros(8, dtype=wp.vec3, device=device)
        self.action = wp.zeros(self.n_act, dtype=float, device=device)

    def reset(self):
        self.x = wp.array(BASE_X, dtype=wp.vec3, device=self.device)
        self.v.zero_()
        self.force.zero_()
        self.action.zero_()

    def step(self, action_np):
        self.action.assign(np.asarray(action_np, dtype=np.float32))

        for _ in range(self.substeps):
            wp.launch(
                compute_force_single,
                dim=8,
                inputs=[
                    self.x, self.v,
                    self.spring_i, self.spring_j,
                    self.rest, self.actuator_index,
                    self.action, self.force,
                    self.n_springs,
                    self.mass, self.k, self.c, self.act_amp,
                    self.ground_k, self.ground_c, self.friction,
                ],
                device=self.device,
            )

            wp.launch(
                integrate_single,
                dim=8,
                inputs=[self.x, self.v, self.force, self.mass, self.dt],
                device=self.device,
            )

    def numpy(self):
        wp.synchronize_device(self.device)
        return self.x.numpy(), self.v.numpy()

    def com(self):
        x, _ = self.numpy()
        return x.mean(axis=0)

## 4. Passive cube: gravity + damping + ground

In [ ]:
sim = WarpCube()
sim.reset()

traj_z = []
for t in range(600):
    sim.step(np.zeros(sim.n_act, dtype=np.float32))
    if t % 10 == 0:
        traj_z.append(sim.com()[2])

x_final, _ = sim.numpy()
print("final COM:", x_final.mean(axis=0))
plot_cube_np(x_final, title="Passive cube after settling")

plt.figure(figsize=(7, 3))
plt.plot(np.arange(len(traj_z))*10, traj_z)
plt.xlabel("simulation step")
plt.ylabel("COM z")
plt.title("Passive settling")
plt.grid(alpha=0.3)
plt.show()

## 5. Hand-designed actuator signal

학습 전에 actuator가 실제로 shape를 바꿀 수 있는지 확인한다.

12개 edge spring에 서로 다른 phase를 준 sinusoid를 사용한다.

In [ ]:
sim.reset()

T = 800
com_traj = []
for t in range(T):
    phase = 2.0 * np.pi * t / 120.0
    offsets = np.linspace(0.0, 2.0*np.pi, sim.n_act, endpoint=False)
    action = np.sin(phase + offsets).astype(np.float32)

    sim.step(action)
    if t % 5 == 0:
        com_traj.append(sim.com().copy())

com_traj = np.array(com_traj)
x_final, _ = sim.numpy()

print("COM displacement:", com_traj[-1] - com_traj[0])
plot_cube_np(x_final, title="Actuated cube")

plt.figure(figsize=(7,3))
plt.plot(com_traj[:,0], label="COM x")
plt.plot(com_traj[:,2], label="COM z")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 6. Stability Lab

아래 세 실험을 수행하라.

1. `k = [80, 180, 400]`
2. `c = [0.2, 2.0, 6.0]`
3. `dt = [0.001, 0.0025, 0.01]`

관찰할 것:
- oscillation
- ground penetration
- numerical explosion
- deformation

### 질문
왜 stiffness가 커질수록 작은 `dt`가 필요할까?

In [ ]:
# 학생 실험용 템플릿
settings = [
    {"k": 80.0,  "c": 2.0, "dt": 0.0025},
    {"k": 180.0, "c": 2.0, "dt": 0.0025},
    {"k": 400.0, "c": 2.0, "dt": 0.0025},
]

for cfg in settings:
    s = WarpCube(k=cfg["k"], c=cfg["c"], dt=cfg["dt"])
    for _ in range(400):
        s.step(np.zeros(s.n_act, np.float32))
    print(cfg, "final COM =", s.com())

다음 notebook에서는 동일한 물리 모델을 **2048개 복제**하고,
하나의 Warp launch로 모든 cube를 GPU에서 동시에 진행시킨다.